# 3차 실험 Colab 스모크 테스트: Ultralytics (YOLO26, RT-DETR)
2차 방법(v1~v4) 하나를 골라 클래스 불균형 대응 조건 세 가지를 학습하고 val로 채점합니다. 스모크 테스트와 본 학습을 같은 코드로 돌립니다.

| 조건 | train 목록 | 손실 가중치 | epoch |
| --- | --- | --- | --- |
| `sampling` | `SAMPLING` 목록 | 없음 | round(E × N ÷ 목록 줄 수) |
| `weight` | 원래 train | `CLS_PW` | E |
| `both` | `SAMPLING` 목록 | `CLS_PW` | round(E × N ÷ 목록 줄 수) |

- E는 비교할 2차 기준 run의 epoch 수(`BASE_EPOCHS`), N은 원래 train 장수입니다. 총 학습량(step 수)을 기준 run과 맞춰서, 샘플링 효과와 학습량 효과가 섞이지 않게 합니다.
- `close_mosaic`(마지막 몇 epoch에 mosaic를 끔)과 `warmup_epochs`도 epoch 단위라서 epoch 수와 같은 비율(N ÷ 목록 줄 수)로 바꿉니다. 그대로 두면 1 epoch 길이가 바뀐 만큼 mosaic 없이 학습하는 step과 warmup step이 기준 run과 달라집니다. (v3 under150: 39 epoch 중 마지막 10 epoch만 mosaic를 끄면 기준 run의 절반) 기준 run의 값은 `BASE_EPOCH_ARGS`에 적습니다.
- 아무것도 하지 않는 기준 조건은 다시 학습하지 않고, 같은 방법·모델의 2차 run을 씁니다.
- 방법, 목록, 가중치 식의 자세한 설명은 [balance.py](../../../src/pill_detection/balance.py) 모듈 docstring에 있습니다.
- 손실 가중치는 Ultralytics 공식 인자 `cls_pw`입니다. Ultralytics는 학습 목록에서 객체를 세므로 `both`의 가중치는 샘플링 후 분포 기준이고, 차이가 `weight`보다 작습니다. (v3, 0.5 기준: 0.48~3.30 → 0.51~1.91)
- **RT-DETR는 `cls_pw`가 손실에 반영되지 않아** `weight`, `both`를 건너뛰고 `sampling`만 돌립니다.

**준비**
- 내 Drive의 `.env`에 `DRIVE_BASE`(pill data 폴더 경로)와 `WANDB_API_KEY`를 적습니다. 키 이름은 [.env.example](../../../.env.example)에서 확인합니다.
- `DRIVE_BASE/data/`에 zip이 있어야 합니다: `kaggle.zip`, `aihub_annotations.zip`, `aihub_images.zip`, 제출할 때만 `test_images.zip`
- 데이터가 팀과 같은지 확인(`make_dataset_lists.py --check`)은 2차 스모크 테스트에서 마쳤다고 봅니다.

**쓰는 법**
1. **각자 바꿀 값** 셀을 채우고 위에서부터 실행합니다. 학습 셀이 `CONDITIONS`의 조건을 차례로 학습하고 val로 채점합니다.
2. 먼저 `SMOKE = True`로 돌려 W&B 기록, Drive 저장, 조건 적용 확인 출력을 봅니다. run 이름이 `smoke_`로 시작하고 태그 `smoke`가 붙으니, 확인이 끝나면 W&B 실행과 Drive 폴더를 지워도 됩니다.
3. `SMOKE = False`로 바꾸고 그 셀부터 다시 실행해 본 학습을 돌립니다. 결과는 `DRIVE_BASE/<SAVE_SUBDIR>/<방법>_<모델>_<조건>`에 남습니다. (예: `v3_yolo26s_rfs`, `v3_yolo26s_cls0.5`, `v3_yolo26s_rfs_cls0.5`)
4. 2차 run과 비교는 [exp3_model_selection.ipynb](exp3_model_selection.ipynb)의 `RUNS`에 이 run 폴더들을 넣어서 봅니다.

> 스모크는 `fraction=0.05`로 목록의 앞 5%만 학습합니다. 본 학습에서는 `fraction`을 쓰지 않습니다. Ultralytics는 목록을 파일명 순서로 정렬해 앞부분만 남기므로, 일부 알약 조합만 학습하게 됩니다.


In [ ]:
# 각자 바꿀 값. 값을 바꾸면 이 셀부터 다시 실행한다
ENV_FILE = '/content/drive/MyDrive/<내 .env 경로>'   # 내 Drive의 .env. DRIVE_BASE, WANDB_API_KEY가 들어 있어야 한다
SAVE_SUBDIR = '<cj_runs, jw_runs 또는 rw_runs>'   # 결과를 저장할 폴더(DRIVE_BASE 아래)
BRANCH = 'feat/41-exp3-balance'                  # balance.py가 있는 브랜치. main에 병합되면 'main'으로 바꾼다

VARIANT = 'v3'          # 기반이 되는 2차 방법: v1, v2, v3, v4 중 하나
BASE_EPOCHS = 20        # 비교할 2차 기준 run의 epoch 수(E)
BASE_EPOCH_ARGS = dict(close_mosaic=10, warmup_epochs=3.0)   # 2차 기준 run의 epoch 단위 설정(Ultralytics 기본값). 조건마다 epoch 수와 같은 비율로 바꿔 쓴다
MODEL = '<yolo26s 또는 rtdetr-l>'   # 3차에 쓸 모델
TRAIN_ARGS = dict(optimizer='SGD', lr0=0.01, imgsz=640, batch=16, seed=0)   # 비교할 2차 실험 run과 같은 값으로 맞춘다

SAMPLING = 'rfs'        # 'rfs'(오버샘플링) 또는 'under150'처럼 'under<클래스별 목표 객체 수>'(언더샘플링)
CLS_PW = 0.5            # 클래스 빈도 손실 가중치 세기(0~1). 0이면 가중치 없음, 1이면 객체 수에 반비례
CONDITIONS = ['sampling', 'weight', 'both']   # 돌릴 조건들. 3개를 모두 넣으면 세 조건을 한 번에 순차적으로 학습돌린다. 하나의 조건만 하고싶으면 ['weight'] 처럼 줄인다
SMOKE = True            # True: 조건마다 1 epoch, train 5%로 기록·저장·조건 적용만 확인한다. 본 학습은 False

In [ ]:
# 세션 시작 (Drive 연결, .env 읽기, 패키지 설치, 레포 받기, W&B 로그인)
import os, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics==8.4.183 pycocotools==2.0.11 wandb==0.31.0 python-dotenv==1.2.4
from dotenv import load_dotenv

if not Path(ENV_FILE).is_file():
    raise FileNotFoundError(f'.env가 없습니다: {ENV_FILE}\n'
                            '[해결] 내 Drive에 .env를 올리고(.env.example 참고), 각자 바꿀 값 셀의 ENV_FILE을 그 경로로 고친 뒤 그 셀부터 다시 실행하세요.')
load_dotenv(ENV_FILE, override=True)
DRIVE_BASE = Path(os.environ.get('DRIVE_BASE', '').strip())
if not os.environ.get('DRIVE_BASE', '').strip() or not DRIVE_BASE.is_dir():
    raise FileNotFoundError(f'DRIVE_BASE가 비어 있거나 Drive에 없는 폴더입니다: "{os.environ.get("DRIVE_BASE", "")}"\n'
                            '[해결] .env에 DRIVE_BASE=<pill data 폴더 경로>를 적으세요. 예) DRIVE_BASE=/content/drive/MyDrive/pill data')
if '<' in SAVE_SUBDIR:
    raise ValueError('SAVE_SUBDIR를 정하지 않았습니다.\n[해결] 각자 바꿀 값 셀의 SAVE_SUBDIR를 내 결과 폴더 이름으로 고치세요.')
if '<' in MODEL:
    raise ValueError("MODEL을 정하지 않았습니다.\n[해결] 각자 바꿀 값 셀의 MODEL을 'yolo26s' 또는 'rtdetr-l'로 고치세요.")
SAVE_DIR = DRIVE_BASE / SAVE_SUBDIR

REPO = '/content/pill-object-detection'
!test -d {REPO} || git clone -q https://github.com/codeit-ai-15th/pill-object-detection.git {REPO}
%cd {REPO}
!git fetch -q origin {BRANCH} && git switch -q -C {BRANCH} FETCH_HEAD && git log -1 --format='%h %s'
sys.path.insert(0, f'{REPO}/src')

import wandb
os.environ['WANDB_ENTITY'] = 'codeit-prj1-team2'   # W&B 팀. .env의 값보다 이 값을 쓴다
wandb.login()   # .env의 WANDB_API_KEY로 로그인한다

In [ ]:
# 데이터 풀기와 2차 학습 폴더 만들기 (세션마다). 이번 세션에서 이미 만들었으면 건너뛴다
DS = f'/content/exp2/{VARIANT}'
if not Path(DS).is_dir():
    for z in ['kaggle', 'aihub_annotations', 'aihub_images']:
        !unzip -q -o "{DRIVE_BASE}/data/{z}.zip" -d {REPO}
    !PYTHONPATH=src python scripts/build_dataset.py --variant {VARIANT} --out {DS}

In [ ]:
# 조건별 train 목록과 epoch. 목록은 입력과 seed가 같으면 늘 같아서 세션마다 다시 만들어도 된다
import pandas as pd
from pill_detection.balance import load_train_objects, repeat_list, undersample_list, write_train_list

objects = load_train_objects(DS)
if SAMPLING == 'rfs':
    files = repeat_list(objects, seed=0)                    # 오버샘플링: 희소 클래스가 든 사진을 더 반복한다
elif SAMPLING.startswith('under') and SAMPLING[5:].isdigit():
    files = undersample_list(objects, int(SAMPLING[5:]))   # 언더샘플링: 클래스별 목표 객체 수까지만 사진을 남긴다
else:
    raise ValueError(f"SAMPLING이 맞지 않습니다: {SAMPLING}\n[해결] 'rfs', 'under150', 'under200' 중 하나를 적으세요.")
info = write_train_list(DS, SAMPLING, files)   # DS/sampling/<SAMPLING>/에 train.txt를 쓴다


def plan(tag, data, n_rows, cls_pw):
    """조건 하나의 학습 계획. epoch 수와 epoch 단위 설정을 같은 비율로 바꿔서 각각의 step 수를 2차 기준 run과 맞춘다."""
    scale = info['n_base'] / n_rows   # 기준 run의 1 epoch ÷ 이 조건의 1 epoch
    return {'name': f'{VARIANT}_{MODEL}_{tag}', 'data': data, 'n_rows': n_rows, 'cls_pw': cls_pw,
            'epochs': round(BASE_EPOCHS * scale),
            'close_mosaic': round(BASE_EPOCH_ARGS['close_mosaic'] * scale),        # mosaic를 끄고 학습하는 step 수를 맞춘다
            'warmup_epochs': round(BASE_EPOCH_ARGS['warmup_epochs'] * scale, 2)}   # warmup step 수를 맞춘다


PLANS = {
    'sampling': plan(SAMPLING, f'{DS}/sampling/{SAMPLING}/data.yaml', info['n_list'], 0.0),
    'weight': plan(f'cls{CLS_PW}', f'{DS}/data.yaml', info['n_base'], CLS_PW),
    'both': plan(f'{SAMPLING}_cls{CLS_PW}', f'{DS}/sampling/{SAMPLING}/data.yaml', info['n_list'], CLS_PW),
}
if set(CONDITIONS) - set(PLANS):
    raise ValueError(f'모르는 조건이 있습니다: {sorted(set(CONDITIONS) - set(PLANS))}\n[해결] CONDITIONS를 sampling, weight, both 중에서 고르세요.')

RUN_CONDITIONS = [c for c in CONDITIONS if not (MODEL.startswith('rtdetr') and PLANS[c]['cls_pw'])]   # RT-DETR는 cls_pw가 손실에 반영되지 않는다
if RUN_CONDITIONS != CONDITIONS:
    print('RT-DETR는 cls_pw가 손실에 반영되지 않아 건너뜁니다:', sorted(set(CONDITIONS) - set(RUN_CONDITIONS)))
display(pd.DataFrame(PLANS).T.loc[RUN_CONDITIONS, ['name', 'n_rows', 'epochs', 'close_mosaic', 'warmup_epochs', 'cls_pw']])

In [ ]:
# 채점 기록 함수: 점수와 조건을 run 폴더(Drive)에 남기고, 학습 때의 W&B 실행 Summary에 exp3/ 점수를 붙인다
import json, shutil
from pill_detection.balance import imbalance_summary


def find_run_id(run_dir):
    """run 폴더의 wandb/ 기록에서 학습 때의 W&B 실행 id를 찾는다. 여러 개면 가장 최근 것."""
    found = max(Path(run_dir, 'wandb').glob('*run-*/run-*.wandb'), key=lambda f: f.stat().st_mtime, default=None)
    if found is None:
        raise FileNotFoundError(f'W&B 기록이 없습니다: {run_dir}/wandb/\n[해결] 학습할 때 W&B 실행의 dir을 run 폴더로 줬는지 확인하세요.')
    return found.stem[4:]


def save_scores(run_dir, scores, condition):
    """val 점수 3개와 희소/빈번 절반 AP(gap)를 출력·저장한다. 2차 run과의 비교는 exp3_model_selection.ipynb가 scores.json으로 한다."""
    summary = {k: scores[k] for k in ('val', 'kaggle', 'test_only')} | imbalance_summary(scores['per_class'], f'{DS}/classes.csv', VARIANT)
    print(Path(run_dir).name, '|', '  '.join(f'{k} {v:.4f}' for k, v in summary.items() if k != 'n_classes'))
    Path(run_dir, 'scores.json').write_text(json.dumps(scores, indent=2), encoding='utf-8')
    Path(run_dir, 'condition.json').write_text(json.dumps(condition, indent=2), encoding='utf-8')
    shutil.copy(f'{DS}/dataset_info.json', run_dir)   # /content는 세션이 끝나면 지워지므로 Drive에 남긴다
    with wandb.init(entity='codeit-prj1-team2', project='pill-object-detection', id=find_run_id(run_dir),
                    resume='allow', dir=str(run_dir)) as run:
        run.summary.update({f'exp3/{k}': v for k, v in summary.items()})
        run.config.update({'exp3': condition})

## 학습과 채점
- `CONDITIONS`의 조건을 차례로 학습하고, `best.pt`로 val을 예측해 채점합니다.
- 다시 실행하면 채점까지 끝난 조건은 건너뛰고, 세션이 끊긴 run은 `last.pt`에서 이어서 학습합니다. (본 학습만. 스모크 run은 실행할 때마다 새 이름입니다)
- 학습 직후 조건 적용 확인을 출력합니다: train 장수가 목록 줄 수와 같은지, `cls_pw`를 준 조건에만 클래스 가중치가 있는지, `close_mosaic`·`warmup_epochs`가 조건 표의 값과 같은지 봅니다. 1 epoch 스모크에서는 두 설정의 효과가 드러나지 않으므로 학습에 들어간 값으로 확인합니다.


In [ ]:
# 조건별 학습과 val 채점
from datetime import datetime
from ultralytics import RTDETR, YOLO, settings
from pill_detection.evaluate import evaluate_val, predictions_from_ultralytics

settings.update({'wandb': True})   # W&B 팀 프로젝트에 기록한다. 팀은 세션 시작 셀에서 정했다
Model = RTDETR if MODEL.startswith('rtdetr') else YOLO


def tag_wandb(tags):
    """첫 epoch 직전에 불린다. W&B 기록이 팀 프로젝트에 시작됐는지 확인하고 태그를 붙인다."""
    if wandb.run is None or wandb.run.entity != 'codeit-prj1-team2':
        raise RuntimeError('W&B 기록이 팀 프로젝트에 시작되지 않았습니다.\n[해결] 런타임을 다시 시작하고 맨 위 셀부터 실행하세요.')
    wandb.run.tags = sorted({*wandb.run.tags, *tags})


def check_applied(trainer, p):
    """조건 적용 확인: train 장수가 목록 줄 수(스모크면 그 5%)와 같고, cls_pw를 준 조건에만 클래스 가중치가 있고,
    close_mosaic, warmup_epochs가 조건 표의 값과 같다."""
    n_train, expected = len(trainer.train_loader.dataset), max(1, round(p['n_rows'] * trainer.args.fraction))
    weighted = getattr(trainer.model, 'class_weights', None) is not None
    schedule = {k: getattr(trainer.args, k) for k in ('close_mosaic', 'warmup_epochs')}   # epoch 단위 설정
    print(f"{trainer.args.name}: train {n_train:,}장 (예상 {expected:,}), 클래스 가중치 {'있음' if weighted else '없음'}, "
          f"close_mosaic {schedule['close_mosaic']} (예상 {p['close_mosaic']}), warmup_epochs {schedule['warmup_epochs']} (예상 {p['warmup_epochs']})")
    if n_train != expected or weighted != bool(p['cls_pw']) or any(schedule[k] != p[k] for k in schedule):
        raise RuntimeError('조건이 학습에 그대로 들어가지 않았습니다.\n[해결] 학습 셀의 data, cls_pw, close_mosaic, warmup_epochs 인자와 조건 셀의 PLANS를 확인하세요.')


for condition in RUN_CONDITIONS:
    p = PLANS[condition]
    epochs = 1 if SMOKE else p['epochs']
    run_dir = SAVE_DIR / (f"smoke_{p['name']}_{datetime.now():%m%d_%H%M}" if SMOKE else p['name'])
    if (run_dir / 'scores.json').exists():
        print(f'{run_dir.name}: 채점까지 끝난 run이라 건너뜁니다.')
        continue
    results_csv = run_dir / 'results.csv'   # epoch가 끝날 때마다 한 줄씩 쌓인다
    if not (results_csv.exists() and len(results_csv.read_text().splitlines()) - 1 >= epochs):
        last = run_dir / 'weights' / 'last.pt'   # 있으면 세션이 끊긴 run이다
        model = Model(last if last.exists() else f'{MODEL}.pt')
        tags = ['exp3', VARIANT, condition] + (['smoke'] if SMOKE else [])
        model.add_callback('on_train_start', lambda trainer: tag_wandb(tags))
        if last.exists():
            model.train(resume=True)
        else:
            model.train(data=p['data'], cls_pw=p['cls_pw'], epochs=epochs, close_mosaic=p['close_mosaic'],
                        warmup_epochs=p['warmup_epochs'], fraction=0.05 if SMOKE else 1.0,
                        project='pill-object-detection', save_dir=str(run_dir), name=run_dir.name, **TRAIN_ARGS)
        check_applied(model.trainer, p)
    results = Model(run_dir / 'weights' / 'best.pt').predict(f'{DS}/images/val', conf=0.001, stream=True, verbose=False)
    save_scores(run_dir, evaluate_val(predictions_from_ultralytics(results), f'{DS}/eval/val_gt.json'),
                {'condition': condition, **p, 'sampling': SAMPLING, 'base_epochs': BASE_EPOCHS, 'smoke': SMOKE})

In [ ]:
# Kaggle 제출 (val로 고른 run만)
SUBMIT_RUN = '<run 폴더 이름>'   # SAVE_DIR 아래 run 폴더. 예) v3_yolo26s_rfs

from pill_detection.evaluate import competition_from_classes, write_submission

!unzip -q -o "{DRIVE_BASE}/data/test_images.zip" -d {REPO}
results = Model(SAVE_DIR / SUBMIT_RUN / 'weights' / 'best.pt').predict(f'{REPO}/data/raw/test_images', conf=0.01, stream=True, verbose=False)
n = write_submission(predictions_from_ultralytics(results), SAVE_DIR / 'submissions' / f'exp3_{SUBMIT_RUN}.csv',
                     competition_from_classes(f'{DS}/classes.csv'))
print(n, '줄')